# 19A Appendix: Similarity Checks, Bias Estimation and the ALS Update


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/19A-Appendix-Recommender-Derivations.ipynb)

This appendix holds the derivations behind [Lecture 19](19-Recommender-Systems.qmd): the Pearson correlation worked on two sparsely rated items and checked against the lecture's item–item table, the Jaccard similarity for binary data, how the bias terms $\mu$, $\alpha_u$, $\beta_i$ are estimated from a sparse ratings matrix and what a negative similarity does to the weighted average, the alternating-least-squares update written out as a ridge regression with a rank-1 example, and the two Netflix Prize extensions beyond biases. The lecture *states* each of these results; this is where they are worked out. The quiz asks you to compute a weighted-average prediction from printed similarities, say why plain SVD does not apply and what a latent dimension means, and read a held-out RMSE table, not to reproduce the updates here.

For where the factors go next, embedding tables and deep recommenders (DLRM), see the optional module [Recommender Systems II](M6-Recommender-Systems-II.qmd).

In [ ]:
#| echo: false
import numpy as np
from IPython.display import display, Markdown

# Similarity in Detail

## Pearson correlation coefficient

The Pearson correlation coefficient, often denoted as $r$, is a measure of the
linear correlation between two variables $X$ and $Y$. It quantifies the degree
to which a linear relationship exists between the variables. The value of $r$
ranges from -1 to 1, where:

- $r = 1$ indicates a perfect positive linear relationship,
- $r = -1$ indicates a perfect negative linear relationship,
- $r = 0$ indicates no linear relationship.

The formula for the Pearson correlation coefficient is:

$$
r = \frac{\sum (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum (X_i - \bar{X})^2 \sum (Y_i - \bar{Y})^2}}
$$

Where:

- $X_i$ and $Y_i$ are the individual sample points,
- $\bar{X}$ and $\bar{Y}$ are the means of the $X$ and $Y$ samples, respectively.

The Pearson correlation coefficient is sensitive to outliers and assumes that the
relationship between the variables is linear and that the data is normally distributed.

In the lecture two items are similar if their ratings are correlated. The catch is that
two items are not rated by the same users, so the correlation is computed only over the
users who rated both:

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L20-corr-support.png)

## Example: two items, twenty-one users

Let's put the ratings in python lists:

In [ ]:
ratings_item_i = [1, np.nan, np.nan, 5, 5, 3, np.nan, np.nan, np.nan, 4, 2, np.nan, np.nan, np.nan, np.nan, 4, np.nan, 5, 4, 1, np.nan]
ratings_item_j = [np.nan, np.nan, 4, 2, 5, np.nan, np.nan, 1, 2, 5, np.nan, np.nan, 2, np.nan, np.nan, 3, np.nan, np.nan, np.nan, 5, 4]

display(Markdown(f'Ratings for item $i$:\n\n{ratings_item_i}'))
display(Markdown(f'Ratings for item $j$:\n\n{ratings_item_j}'))

Let's drop the non-common ratings:

In [ ]:
# Keep only the positions where both items have a rating
filtered_ratings_item_i = [rating_i for rating_i, rating_j in zip(ratings_item_i, ratings_item_j) if not np.isnan(rating_i) and not np.isnan(rating_j)]
filtered_ratings_item_j = [rating_j for rating_i, rating_j in zip(ratings_item_i, ratings_item_j) if not np.isnan(rating_i) and not np.isnan(rating_j)]

display(Markdown(f'Common ratings for item $i$: {filtered_ratings_item_i}'))
display(Markdown(f'Common ratings for item $j$: {filtered_ratings_item_j}'))

Only five of the twenty-one users rated both. Now we can compute the Pearson correlation coefficient:

In [ ]:
rho = np.corrcoef(filtered_ratings_item_i, filtered_ratings_item_j)[0,1]
display(Markdown(f'Pearson correlation coefficient: {rho:0.2f}'))

Which is a _moderate negative correlation_, meaning that as item $i$ gets rated higher,
item $j$ gets rated lower. Notice that it rests on five people: that is the sparsity
problem showing up inside the similarity itself.

## Checking the worked example

The same computation reproduces the similarity table of the lecture's item–item example
(items are rows, users are columns; user 5's rating of item 1 is the one being predicted).

In [ ]:
nan = np.nan
R_toy = np.array([            # rows = items 1..6, columns = users 1..12
    [1, nan, 3, nan, nan, 5, nan, nan, 5, nan, 4, nan],
    [nan, nan, 5, 4, nan, nan, 4, nan, nan, 2, 1, 3],
    [2, 4, nan, 1, 2, nan, 3, nan, 4, 3, 5, nan],
    [nan, 2, 4, nan, 5, nan, nan, 4, nan, nan, 2, nan],
    [nan, nan, 4, 3, 4, 2, nan, nan, nan, nan, 2, 5],
    [1, nan, 3, nan, 3, nan, nan, 2, nan, nan, 4, nan]])

user = 4                      # user 5, zero-indexed
for j in np.where(~np.isnan(R_toy[:, user]))[0]:
    both = ~np.isnan(R_toy[0]) & ~np.isnan(R_toy[j])
    s = np.corrcoef(R_toy[0, both], R_toy[j, both])[0, 1]
    raters = ', '.join(str(k + 1) for k in np.where(both)[0])
    print(f'item {j + 1}: co-raters {raters}, s_1{j + 1} = {s:5.2f}')

With $k = 2$ the two most similar items are 3 and 6, and the prediction is the
similarity-weighted average $(0.84 \cdot 2 + 1.0 \cdot 3)/(0.84 + 1.0) = 2.54$.

## The user–user mirror

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L20-u-u-cf-1.png)

The user–user mirror on the same matrix: the users who rated item 1 and share at least
three items with user 5 (with two, a correlation is always $\pm 1$, as for item 4 above)
are users 3 and 11. User 3 correlates positively ($0.87$ over items 4, 5, 6) and rated
item 1 a 3; user 11 correlates negatively ($-0.95$) and is dropped. User–user CF
predicts 3. Two flavours, same matrix, slightly different answers: $2.54$ and $3$.

## Similarity for binary data

In some cases we will need to work with binary $r_{ui}$: for example, purchase histories
on an e-commerce site, or clicks on an ad. In this case, an appropriate replacement for
Pearson $r$ is the **Jaccard similarity coefficient** or **Intersection over Union**,

$$
J_{Sim}(\mathbf{x}, \mathbf{y}) = \frac{|\mathbf{x} \cap \mathbf{y}|}{|\mathbf{x} \cup \mathbf{y}|},
$$

the number of users who interacted with both items divided by the number who interacted
with either. See the lecture on [similarity measures](02-Distances-Timeseries.qmd#norms).

# Bias Estimation in Detail

## Representing biases

Some items are significantly higher or lower rated, and some users rate substantially
higher or lower in general. These properties interfere with similarity assessment, so
the lecture forms a per-user, per-item bias

$$
b_{ui} = \mu + \alpha_u + \beta_i
$$

where

* $b_{ui}$ is the bias of user $u$ for item $i$,
* $\mu$ is the global average rating across all items and users,
* $\alpha_u$ is the offset of user $u$ and
* $\beta_i$ is the offset of item $i$.

If we gather all these elements together we can form:

* $\boldsymbol{\alpha}$ an $n\times 1$ vector of per-user offsets, and
* $\boldsymbol{\beta}$ an $m\times 1$ vector of per-item offsets.

## Estimating biases

How can we estimate the parameters $\boldsymbol{\alpha}$, $\boldsymbol{\beta}$, and $\mu$?

Let's assume for a minute that we had a fully-dense matrix of ratings $R$. Recall that
each of the $m$ rows of $R$ represents an item and each of the $n$ columns represents a
user.

One way to do this is to minimize the squared error between the ratings and the biases,
and include a regularization term to minimize the magnitude of the biases:

$$
\min_{\boldsymbol{\alpha},\boldsymbol{\beta},\mu}
\Vert R - (\mathbf{1}\boldsymbol{\alpha}^T + \boldsymbol{\beta}\mathbf{1}^T + \mu1)\Vert^2
+ \lambda(\Vert\boldsymbol{\alpha}\Vert^2 + \Vert\boldsymbol{\beta}\Vert^2).
$$

* Here, bold-faced $\mathbf{1}$ represents appropriately sized vectors of ones,
  and non-boldfaced $1$ is an $m\times n$ matrix of ones.
* So $\mathbf{1}\boldsymbol{\alpha}^T$ is an $m\times n$ matrix in which each column is constant, equal to that user's offset $\alpha_u$.
* Similarly, $\boldsymbol{\beta}\mathbf{1}^T$ is an $m\times n$ matrix in which each row is constant, equal to that item's offset $\beta_i$.
* And $\mu1$ is an $m\times n$ matrix where each element is $\mu$.

There is a simple strategy for solving it. Assume we hold $\beta\mathbf{1}^T$ and $\mu1$
constant. Then the remaining problem is

$$
\min_{\alpha} \Vert R - \mathbf{1}\alpha^T \Vert^2 + \lambda \Vert\alpha\Vert^2,
$$

which (for each column of $R$) is a standard regularized least squares problem
solved via [Ridge regression](./11B-Regularization.qmd#ridge-regression):
$\hat{\boldsymbol{\beta}} = (X^TX + cI)^{-1}X^T\mathbf{y}$.

## Solving by alternation

This problem is __convex__, in fact jointly convex in $\mu$, $\alpha$ and $\beta$,
because the prediction is linear in all three: it is one big least-squares problem.
Solving for one block at a time is just convenient. (The matrix-factorization problem
below is *not* jointly convex.)

The strategy for solving is:

1. Hold $\alpha$ and $\beta$ constant, solve for $\mu$.
2. Hold $\alpha$ and $\mu$ constant, solve for $\beta$.
3. Hold $\beta$ and $\mu$ constant, solve for $\alpha$.

Each of the three steps will reduce the overall error. As a result, we iterate over them
until convergence.

The last issue is that the matrix $R$ is not dense: in reality we only have a small
subset of its entries. We simply need to adapt the least-squares solution to only
consider the entries in $R$ that we know. As a result, the actual calculation is as
follows.

Step 1:

$$
\mu = \frac{\sum_{(u, i) \in R} (r_{ui} - \alpha_u - \beta_i)}{|R|}
$$

Step 2:

$$
\alpha_u = \frac{\sum_{i \in R(u)}(r_{ui} - \mu - \beta_i)}{\lambda + |R(u)|}
$$

Step 3:

$$
\beta_i = \frac{\sum_{u \in R(i)}(r_{ui} - \mu - \alpha_u)}{\lambda + |R(i)|}
$$

Step 4: If not converged, go to Step 1.

Here $i \in R(u)$ means the set of items rated by user $u$ and $u \in R(i)$ means
the set of users who have rated item $i$ and $|R(u)|$ is the number of ratings.

Notice the $\lambda$ in the denominator. A user with only two ratings gets their offset
pulled toward zero; a user with five hundred ratings barely notices $\lambda$. That is
shrinkage, and it appears again in the ALS update below.

## Bias-corrected similarity and prediction

Now that we have learned the biases, we can do a better job of estimating correlation:

$$
\hat{\rho}_{ij} = \frac{\sum_{u\in U(i,j)}(r_{ui} - b_{ui})(r_{uj}-b_{uj})}
{\sqrt{\sum_{u\in U(i,j)}(r_{ui} - b_{ui})^2\sum_{u\in U(i,j)}(r_{uj}-b_{uj})^2}},
$$

where

* $b_{ui} = \mu + \alpha_u + \beta_i$, and
* $U(i,j)$ are the users who have rated both $i$ and $j$.

When $b_{ui}$ is replaced by user $u$'s mean rating, this is the
**adjusted cosine similarity** you will use in the activity.

And using biases we can also do a better job of estimating ratings:

$$
\hat{r}_{ui} = b_{ui} + \frac{\sum_{j \in n_k(i, u)} s_{ij}(r_{uj} - b_{uj})}{\sum_{j \in n_k(i, u)} s_{ij}},
$$

where

* $b_{ui} = \mu + \alpha_u + \beta_i$,
* $n_k(i, u)$ are the $k$ nearest neighbors to $i$ that were rated by user $u$ and
* $s_{ij}$ is the similarity between items $i$ and $j$, estimated as above.

## Negative similarity scores

When using the correlation coefficient as the similarity score, negative values have
important implications for the weighted calculation above. When $s_{ij}$ is **negative**
(negative correlation), here's what happens:

1. **Numerator**: The term $s_{ij}(r_{uj} - b_{uj})$ becomes negative when $s_{ij} < 0$. This means:
   - If user $u$ rated item $j$ **above** its bias ($r_{uj} - b_{uj} > 0$), the negative similarity will push the prediction for item $i$ **down**
   - If user $u$ rated item $j$ **below** its bias ($r_{uj} - b_{uj} < 0$), the negative similarity will push the prediction for item $i$ **up**

   This makes intuitive sense: if items $i$ and $j$ are negatively correlated, then liking one means you're likely to dislike the other.

2. **Denominator**: When $s_{ij}$ is negative, it contributes a negative value to the sum in the denominator. This can potentially lead to issues:
   - If all similarities are negative, the denominator becomes negative
   - If there's a mix of positive and negative similarities, they might partially cancel out, potentially making the denominator close to zero or even zero

## Common solutions

To address these issues, practitioners typically use one of these approaches:

1. **Use only positive similarities**: Filter out items with negative correlation (only use $k$-NN where $s_{ij} > 0$)

2. **Use absolute values in denominator**: $\displaystyle\hat{r}_{ui} = b_{ui} + \frac{\sum_{j \in n_k(i, u)} s_{ij}(r_{uj} - b_{uj})}{\sum_{j \in n_k(i, u)} |s_{ij}|}$

The first approach (filtering to positive similarities) is most common because negatively correlated items are typically less informative for prediction than positively correlated items in most domains.

You will sometimes see a third option, rescaling to $[0, 1]$ with $s'_{ij} = (s_{ij} + 1)/2$.
Avoid it: it gives an uncorrelated neighbour weight $0.5$ and an anti-correlated one
weight near $0$, so it rewards neighbours that carry no information.

# The ALS Update in Detail

## Why not the SVD

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L20-mf-1.png)

In the lecture's figure the factorization is rank 3, i.e., low-rank. So we want
something like an SVD. (Recall that the SVD gives us the most-accurate-possible low-rank
factorization of a matrix.)


However, we can't use the SVD algorithm directly, because we don't know all the entries in $R$.

> Indeed, the unseen entries in $R$ are exactly what we want to predict.

## The objective

Here is what we want to solve:

$$
\min_{U,V} \sum_{(u,i)\in S} \left(r_{ui} - \mathbf u_u^T \mathbf v_i\right)^2 + \lambda(\Vert U\Vert^2 + \Vert V\Vert^2),
$$

or in matrix form $\min_{U,V} \Vert (R - UV^T)_S\Vert^2 + \lambda(\Vert U\Vert^2 + \Vert V\Vert^2)$,

where:

* $U$ is the $n\times k$ **users** matrix: row $u$ is $\mathbf u_u^T$,
* $V$ is the $m\times k$ **items** matrix: row $i$ is $\mathbf v_i^T$,
* $k$ is the rank of the factorization and dimensionality of the latent space, and
* in the matrix form $R$ is laid out users × items ($n \times m$), as in the lecture's
  code; the figure draws its transpose, $R^T \approx VU^T$.

The $(\cdot)_S$ notation means that we are only considering the _subset_ of
matrix entries that correspond to known reviews (the set $S$).

Note that as usual, we add $\ell_2$ penalization to avoid overfitting
([Ridge regression](./11B-Regularization.qmd#ridge-regression)).

## Biconvexity and alternating least squares

This problem is __not__ convex in $U$ and $V$ together, because of the product $UV^T$.
It is __biconvex__: convex in $U$ when $V$ is held fixed, and convex in $V$ when $U$ is
held fixed. In particular, if we hold either $U$ or $V$ constant, then the result is a
simple ridge regression.

So one commonly used algorithm for this problem is called __alternating least squares (ALS):__

1. Hold $U$ constant, and solve for $V$
2. Hold $V$ constant, and solve for $U$
3. If not converged, go to Step 1.

Every half-step can only lower the objective, so the iteration converges, though to a
local optimum of the joint problem, not necessarily the global one.

## The ALS update

Hold $V$ fixed. The objective splits into one small problem per user, because each
user's vector only appears in that user's ratings. For user $u$, let $J_u$ be the items
she rated, $V_{J_u}$ their rows of $V$, and $\mathbf r_{u,J_u}$ her ratings:

$$
\min_{\mathbf u_u} \Vert \mathbf r_{u,J_u} - V_{J_u}\mathbf u_u \Vert^2 + \lambda \Vert \mathbf u_u\Vert^2
\quad\Longrightarrow\quad
\mathbf u_u = \left(V_{J_u}^T V_{J_u} + \lambda I\right)^{-1} V_{J_u}^T\, \mathbf r_{u,J_u}.
$$

This is the ridge formula $\hat{\boldsymbol\beta} = (X^TX + cI)^{-1}X^T\mathbf y$ with
$X = V_{J_u}$ (the item vectors are the features), $\mathbf y = \mathbf r_{u,J_u}$
(her ratings) and $c = \lambda$.

The other half-step is the mirror image: for item $i$, regress the ratings it received
on the factors of the users who rated it.

`solve_U` in `scripts/recommender_als.py` is exactly this line, once per user:
`np.linalg.solve(V[W[i]].T.dot(V[W[i]]) + I, V[W[i]].T.dot(x_i[W[i]]))`.

With $J_u$ empty (a user who has rated nothing) the formula gives
$\mathbf u_u = (\lambda I)^{-1}\mathbf 0 = \mathbf 0$: this is the lecture's cold-start
observation.

## A rank-1 example

Take $k = 1$, so every factor is a number. Fixed item factors:
$v_1 = 1$, $v_2 = 2$, $v_3 = 1.75$.

User $u$ rated $r_{u1} = 2$ and $r_{u2} = 5$, and has not rated item 3.

$$
J(u_u) = (2 - u_u)^2 + (5 - 2u_u)^2 + \lambda u_u^2
\quad\Longrightarrow\quad
u_u = \frac{1\cdot 2 + 2 \cdot 5}{1^2 + 2^2 + \lambda} = \frac{12}{5 + \lambda}.
$$

| | $u_u$ | $\hat r_{u1}$ | $\hat r_{u2}$ | $\hat r_{u3}$ |
|---|---|---|---|---|
| $\lambda = 0$ | 2.4 | 2.4 | 4.8 | 4.2 |
| $\lambda = 1$ | 2.0 | 2.0 | 4.0 | 3.5 |

The penalty pulls the factor toward 0, and every prediction shrinks with it:
**shrinkage**. It costs some fit on the two ratings she gave, and protects against
trusting a factor estimated from very few ratings. (Try $\lambda = 5$: $u_u = 1.2$.)

The same numbers through the ridge formula:

In [ ]:
V_rated = np.array([[1.0], [2.0]])   # item factors of the items she rated
r_u = np.array([2.0, 5.0])           # her ratings
for lam in [0, 1]:
    u_u = np.linalg.solve(V_rated.T @ V_rated + lam * np.eye(1), V_rated.T @ r_u)
    print(f'lambda = {lam}: u_u = {u_u[0]:.2f}, prediction for item 3 = {1.75 * u_u[0]:.2f}')

Same line of algebra, whether the factor has one number or twenty.

# Beyond ALS: the Netflix Prize Extensions

The lecture's last improvement put the biases inside the factorization. The winning
Netflix Prize team made two further additions [@koren2009matrix; @koren2009collaborative].

## Who rated what

In reality, ratings are not provided __at random.__ The mere fact that someone chose to
rate a movie says something about them, whatever the rating was. Take note of which users
rated the same movies (à la CF) and use this information: on the left, the ratings; on
the right, the same matrix reduced to who rated what, a one wherever there is a rating
and a zero elsewhere. That binary matrix has no holes, and the models learn from it too.

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L20-netflix-2.png)

On the error plot, the arrows mark the jump from the bias model to SVD++, the version
that uses who rated what. Notice the horizontal axis: millions of parameters, running
into the billions for the largest models.

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L20-netflix-3.png)

## Ratings change over time

Older movies tend to get higher ratings! One explanation is that people who seek out
older films already expect to like them.

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L20-netflix-4.png)

If we add time-varying biases, and let each **user's** taste drift (the item
factors stay fixed) [@koren2009collaborative]:

$$
\min_{U, V} \sum_{(u, i)\in S}\left(r_{ui} - (\mu + \alpha_u(t) + \beta_i(t) + \mathbf u_u(t)^T\mathbf v_i)\right)^2
+ \lambda(\Vert U\Vert^2 + \Vert V\Vert^2 + \Vert \alpha\Vert^2 + \Vert \beta \Vert^2)
$$

we see further improvements in accuracy:

![](https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/figs/L20-netflix-5.png)

With the time and implicit-feedback terms the model is no longer bilinear in two
blocks of parameters, so the Netflix teams fit it with stochastic gradient descent
([Lecture 16](16-Neural-Networks.qmd)) rather than ALS.

## Going further

The optional module [Recommender Systems II](M6-Recommender-Systems-II.qmd) treats $U$
and $V$ as embedding tables, a lookup from an ID to a learned vector, and shows how modern
deep recommenders (DLRM) keep exactly this core and add neural networks around it.

## Bibliography
